# Supersuit + Stable Baselines3 Training - MultiCarRacing with 2 Agents

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import VecMonitor, VecFrameStack, VecTransposeImage, DummyVecEnv, VecNormalize
import supersuit as ss
from multi_car_racing import parallel_env

In [ ]:
env = parallel_env(num_agents=2, render_mode=None, use_ego_color=True, continuous=False, max_episode_steps=1000, ctde=True)
print(env.reset()[0].keys())
env = ss.pettingzoo_env_to_vec_env_v1(env)
env = ss.concat_vec_envs_v1(env, num_vec_envs=4, num_cpus=1, base_class='stable_baselines3')
env = VecMonitor(env)
env = VecFrameStack(env, n_stack=4)
env = VecTransposeImage(env)
env = VecNormalize(env, norm_obs=False, norm_reward=False, training=False)
print(type(env))
print(env.reset().shape)

print(env.observation_space)
print(env.action_space)

In [ ]:
model = DQN("CnnPolicy", env, verbose=1, tensorboard_log="./dqn_tensorboard/", buffer_size=100000, device="mps")
model.learn(total_timesteps=1000000, progress_bar=True)
model.save("dqn_multi_car_racing")

In [ ]:
from time import sleep
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import VecMonitor, VecFrameStack, VecTransposeImage, VecNormalize
import supersuit as ss
from multi_car_racing import parallel_env
import numpy as np
# 1) create the parallel PettingZoo env with human render
par_env = parallel_env(num_agents=4, render_mode="human", use_ego_color=True, human_show_team_colors=True, continuous=False, auto_reset=True, team_ids=[0,0,1,1])
# 2) apply the same wrappers you used for training (order matters)
# 3) convert to a single Markov VecEnv (do NOT use multi-cpu concat here)
env = ss.pettingzoo_env_to_vec_env_v1(par_env)
env = ss.concat_vec_envs_v1(env, num_vec_envs=1, num_cpus=1, base_class='stable_baselines3')
env = VecMonitor(env)
env = VecFrameStack(env, n_stack=4)
env = VecNormalize(env, norm_obs=False, norm_reward=False, training=False)
env = VecTransposeImage(env)
# 4) load the model (or use existing 'model')
model = DQN.load("../experiments/teams_iql_cnn_seed42_0/best_model/best_model.zip", env=env)

# 5) run deterministic policy in a loop and render
vec_env = model.get_env()
obs = vec_env.reset()
done = False
while not np.all(done):
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action)
    vec_env.render("human")

env.close()